# SMS Spam Detection using Logistic Regression

## Student Details
Name: Linoy O.
ID last 4 digits: 8874

Name: Elay S.
ID last 4 digits: 7500

## Problem Description
In this project, we solve a binary text classification problem.
The goal is to classify SMS messages as either spam or ham.

The input data contains text messages and their labels:
- ham: regular message, not spam
- spam: unwanted spam message

Since machine learning models cannot work directly with raw text, we convert the messages into numerical features.

We use Logistic Regression as the classification algorithm.
The main evaluation metric is F1-score for the spam class, because spam is the important class in this task.

## Dataset
Dataset name: SMS Spam Collection Dataset  
Dataset source: Kaggle  
Dataset URL: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

## Tools and Assistance

We used ChatGPT as a learning assistant during this assignment.

## AI Prompt Used

Prompt:
"Help us understand how to organize a machine learning notebook for SMS spam classification using Logistic Regression. 
Explain the general workflow, including text preprocessing, TF-IDF feature extraction, train/test evaluation, F1-score for the spam class, GridSearchCV, and final model explanation."

Purpose:
We used this prompt to receive general guidance about the structure and requirements of the machine learning workflow.
The assistance was used to better understand the steps of the project, the role of each stage, and how to explain the process clearly.

We did not use ChatGPT to automatically generate the full notebook or submit code without understanding it.
The code was reviewed, adapted, executed, and explained by us as part of this assignment.
All outputs, experiments, explanations, and conclusions in this notebook are based on our own work and execution.

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## Loading the Dataset

In this section, we load the SMS spam dataset and display the first rows.
The dataset contains SMS messages and their labels.ls.

In [2]:
sms_data = pd.read_csv("spam.csv", encoding="latin-1")

sms_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
sms_data = sms_data[["v1", "v2"]]

sms_data = sms_data.rename(columns={
    "v1": "category",
    "v2": "sms_text"
})

sms_data.head()

,category,sms_text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Dataset Overview

We check the dataset size, missing values, and class distribution.
This helps us understand whether the dataset is balanced or imbalanced.

In [4]:
print("Dataset shape:")
print(sms_data.shape)

print("\nMissing values:")
print(sms_data.isnull().sum())

print("\nClass distribution:")
print(sms_data["category"].value_counts())

Dataset shape:
(5572, 2)

Missing values:
category    0
sms_text    0
dtype: int64

Class distribution:
category
ham     4825
spam     747
Name: count, dtype: int64


The dataset is imbalanced because most messages are ham and fewer messages are spam.
Therefore, accuracy alone is not enough.
For this reason, we focus mainly on F1-score for the spam class.

In [5]:
sms_data["target"] = sms_data["category"].map({
    "ham": 0,
    "spam": 1
})

sms_data.head()

,category,sms_text,target
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## Train-Test Split

The dataset was provided as a single CSV file and not as separate train and test files.
Therefore, we split it once into a training set and a testing set using stratified train-test split.

The training set is used to train the model.
The testing set is used only for final evaluation.
After this split, we do not split the test set again.

In [6]:
messages = sms_data["sms_text"]
labels = sms_data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    messages,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

In [7]:
train_preview = pd.DataFrame({
    "sms_text": X_train,
    "label": y_train
})

train_preview.head()

,sms_text,label
1266,\Hey sorry I didntgive ya a a bellearlier hunny,0
1851,Dunno da next show aft 6 is 850. Toa payoh got...,0
310,Today is ACCEPT DAY..U Accept me as? Brother S...,0
3708,Ok.ok ok..then..whats ur todays plan,0
1020,Good afternoon on this glorious anniversary da...,0


In [8]:
test_preview = pd.DataFrame({
    "sms_text": X_test,
    "label": y_test
})

test_preview.head()

,sms_text,label
1699,Ok...,0
48,"Yeah hopefully, if tyler can't do it I could m...",0
417,FREE entry into our å£250 weekly competition j...,1
5052,Lmao you know me so well...,0
5298,I.ll hand her my phone to chat wit u,0


## Text Normalization

Before converting the SMS messages into numerical features, we normalize the text.
The normalization includes:
- converting text to lowercase
- replacing URLs with the word url
- replacing numbers with the word number
- removing special characters
- removing extra spaces

In [9]:
def normalize_sms_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " url ", text)
    text = re.sub(r"\d+", " number ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [10]:
X_train_normalized = X_train.apply(normalize_sms_text)
X_test_normalized = X_test.apply(normalize_sms_text)

In [11]:
normalization_examples = pd.DataFrame({
    "original_message": X_train.head(3).values,
    "normalized_message": X_train_normalized.head(3).values
})

normalization_examples

,original_message,normalized_message
0,\Hey sorry I didntgive ya a a bellearlier hunny,hey sorry i didntgive ya a a bellearlier hunny
1,Dunno da next show aft 6 is 850. Toa payoh got...,dunno da next show aft number is number toa pa...
2,Today is ACCEPT DAY..U Accept me as? Brother S...,today is accept day u accept me as brother sis...


## Feature Engineering with TF-IDF

Logistic Regression cannot work directly with raw text.
Therefore, we use TF-IDF to convert each SMS message into a numeric vector.

TF-IDF gives higher importance to words that are meaningful in a message and less common across all messages.

In [12]:
tfidf_extractor = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    ngram_range=(1, 1)
)

X_train_tfidf = tfidf_extractor.fit_transform(X_train_normalized)
X_test_tfidf = tfidf_extractor.transform(X_test_normalized)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (4457, 3000)
Test TF-IDF shape: (1115, 3000)


## Feature Engineering Examples

In this section, we show how several SMS messages are represented after TF-IDF transformation.
Only the non-zero TF-IDF values are shown.

In [13]:
def explain_vectorized_message(vectorized_matrix, row_number, vectorizer, top_n=10):
    feature_names = vectorizer.get_feature_names_out()
    row_values = vectorized_matrix[row_number].toarray()[0]
    
    non_zero_positions = np.where(row_values > 0)[0]
    
    explanation_table = pd.DataFrame({
        "feature": feature_names[non_zero_positions],
        "tfidf_score": row_values[non_zero_positions]
    })
    
    return explanation_table.sort_values(
        by="tfidf_score",
        ascending=False
    ).head(top_n)

In [14]:
for i in range(3):
    print("Train example", i + 1)
    print(X_train_normalized.iloc[i])
    display(explain_vectorized_message(X_train_tfidf, i, tfidf_extractor))

Train example 1
hey sorry i didntgive ya a a bellearlier hunny


,feature,tfidf_score
1,hunny,0.672760
3,ya,0.477442
0,hey,0.409612
2,sorry,0.389438


Train example 2
dunno da next show aft number is number toa payoh got number


,feature,tfidf_score
5,toa,0.539726
4,number,0.447671
0,aft,0.426982
2,dunno,0.394549
1,da,0.309889
3,got,0.272531


Train example 3
today is accept day u accept me as brother sister lover dear number best number clos number lvblefrnd jstfrnd cutefrnd lifpartnr belovd swtheart bstfrnd no rply means enemy


,feature,tfidf_score
0,accept,0.451646
6,cutefrnd,0.232289
18,swtheart,0.232289
13,lvblefrnd,0.232289
11,lifpartnr,0.232289
10,jstfrnd,0.232289
5,clos,0.232289
4,bstfrnd,0.232289
9,enemy,0.225823
1,belovd,0.225823


In [15]:
for i in range(3):
    print("Test example", i + 1)
    print(X_test_normalized.iloc[i])
    display(explain_vectorized_message(X_test_tfidf, i, tfidf_extractor))

Test example 1
ok


,feature,tfidf_score
0,ok,1.0


Test example 2
yeah hopefully if tyler can t do it i could maybe ask around a bit


,feature,tfidf_score
2,hopefully,0.496201
4,tyler,0.496201
3,maybe,0.393724
1,bit,0.367064
0,ask,0.330011
5,yeah,0.330011


Test example 3
free entry into our number weekly competition just text the word win to number now number t c url


,feature,tfidf_score
0,competition,0.428545
4,number,0.378753
1,entry,0.355132
7,weekly,0.352312
9,word,0.326466
8,win,0.292822
6,url,0.273395
5,text,0.244524
2,free,0.229386
3,just,0.207780


## Logistic Regression Explanation

Logistic Regression is a supervised learning algorithm used for classification.

In this project, the model receives TF-IDF features that represent the SMS messages.
The model learns a weight for each word or phrase.
Features with positive weights increase the probability that a message is spam.
Features with negative weights make the message more likely to be ham.

The model outputs a prediction for each message: either ham or spam.

In [16]:
def create_logistic_classifier(c_value=1.0, use_balanced_weights=True):
    selected_class_weight = "balanced" if use_balanced_weights else None
    
    classifier = LogisticRegression(
        C=c_value,
        class_weight=selected_class_weight,
        max_iter=1200,
        random_state=7
    )
    
    return classifier

In [17]:
def fit_sms_classifier(classifier, train_features, train_labels):
    classifier.fit(train_features, train_labels)
    return classifier

In [18]:
def predict_sms_labels(trained_classifier, message_features):
    predicted_labels = trained_classifier.predict(message_features)
    return predicted_labels

In [19]:
logistic_spam_model = create_logistic_classifier(
    c_value=1.0,
    use_balanced_weights=True
)

logistic_spam_model = fit_sms_classifier(
    logistic_spam_model,
    X_train_tfidf,
    y_train
)

In [20]:
basic_predictions = predict_sms_labels(
    logistic_spam_model,
    X_test_tfidf
)

## Model Evaluation

The main metric is F1-score for the spam class.
We also show the confusion matrix and classification report.

In [21]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, basic_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    basic_predictions,
    target_names=["ham", "spam"]
))

spam_f1_basic = f1_score(y_test, basic_predictions, pos_label=1)
print("F1 score for spam:", spam_f1_basic)

Confusion Matrix:
[[939  27]
 [  8 141]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.97      0.98       966
        spam       0.84      0.95      0.89       149

    accuracy                           0.97      1115
   macro avg       0.92      0.96      0.94      1115
weighted avg       0.97      0.97      0.97      1115

F1 score for spam: 0.889589905362776


In [22]:
basic_metrics = pd.DataFrame({
    "metric": [
        "accuracy",
        "precision_spam",
        "recall_spam",
        "f1_spam"
    ],
    "score": [
        accuracy_score(y_test, basic_predictions),
        precision_score(y_test, basic_predictions, pos_label=1),
        recall_score(y_test, basic_predictions, pos_label=1),
        f1_score(y_test, basic_predictions, pos_label=1)
    ]
})

basic_metrics

,metric,score
0,accuracy,0.968610
1,precision_spam,0.839286
2,recall_spam,0.946309
3,f1_spam,0.889590


## First 5 Test Predictions

The table below shows the first five predictions on the test set.

In [23]:
first_predictions_table = pd.DataFrame({
    "sms_message": X_test.iloc[:5].values,
    "true_value": y_test.iloc[:5].values,
    "predicted_value": basic_predictions[:5]
})

first_predictions_table["true_label"] = first_predictions_table["true_value"].replace({
    0: "ham",
    1: "spam"
})

first_predictions_table["predicted_label"] = first_predictions_table["predicted_value"].replace({
    0: "ham",
    1: "spam"
})

first_predictions_table

,sms_message,true_value,predicted_value,true_label,predicted_label
0,Ok...,0,0,ham,ham
1,"Yeah hopefully, if tyler can't do it I could m...",0,0,ham,ham
2,FREE entry into our å£250 weekly competition j...,1,1,spam,spam
3,Lmao you know me so well...,0,0,ham,ham
4,I.ll hand her my phone to chat wit u,0,0,ham,ham


## Manual Feature Engineering Experiments

In this section, we compare several feature engineering and Logistic Regression configurations.
We compare CountVectorizer and TF-IDF, unigrams and bigrams, and different C values.

These manual experiments are shown for comparison and understanding.
The final model selection is done using GridSearchCV with 5-fold cross validation on the training set.
The test set is used only for the final evaluation of the selected model.

In [24]:
experiment_configurations = [
    {
        "experiment_name": "CountVectorizer_unigrams_C1",
        "vectorizer": CountVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 1)),
        "C": 1.0
    },
    {
        "experiment_name": "TFIDF_unigrams_C1",
        "vectorizer": TfidfVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 1)),
        "C": 1.0
    },
    {
        "experiment_name": "TFIDF_bigrams_C1",
        "vectorizer": TfidfVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 2)),
        "C": 1.0
    },
    {
        "experiment_name": "TFIDF_bigrams_C05",
        "vectorizer": TfidfVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 2)),
        "C": 0.5
    },
    {
        "experiment_name": "TFIDF_bigrams_C2",
        "vectorizer": TfidfVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 2)),
        "C": 2.0
    }
]

In [25]:
manual_experiment_results = []

for config in experiment_configurations:
    current_vectorizer = config["vectorizer"]
    
    train_features = current_vectorizer.fit_transform(X_train_normalized)
    test_features = current_vectorizer.transform(X_test_normalized)
    
    current_model = create_logistic_classifier(
        c_value=config["C"],
        use_balanced_weights=True
    )
    
    current_model.fit(train_features, y_train)
    current_predictions = current_model.predict(test_features)
    
    manual_experiment_results.append({
        "experiment": config["experiment_name"],
        "vectorizer_type": type(current_vectorizer).__name__,
        "C": config["C"],
        "accuracy": accuracy_score(y_test, current_predictions),
        "precision_spam": precision_score(y_test, current_predictions, pos_label=1),
        "recall_spam": recall_score(y_test, current_predictions, pos_label=1),
        "f1_spam": f1_score(y_test, current_predictions, pos_label=1)
    })

manual_results_df = pd.DataFrame(manual_experiment_results)
manual_results_df.sort_values(by="f1_spam", ascending=False)

,experiment,vectorizer_type,C,accuracy,precision_spam,recall_spam,f1_spam
0,CountVectorizer_unigrams_C1,CountVectorizer,1.0,0.984753,0.958333,0.926174,0.941980
4,TFIDF_bigrams_C2,TfidfVectorizer,2.0,0.974888,0.871166,0.953020,0.910256
1,TFIDF_unigrams_C1,TfidfVectorizer,1.0,0.968610,0.839286,0.946309,0.889590
2,TFIDF_bigrams_C1,TfidfVectorizer,1.0,0.966816,0.833333,0.939597,0.883281
3,TFIDF_bigrams_C05,TfidfVectorizer,0.5,0.965022,0.827381,0.932886,0.876972


## Grid Search with 5-Fold Cross Validation

In this section, we use GridSearchCV to test multiple combinations of feature engineering parameters and Logistic Regression hyperparameters.
The best configuration is selected according to the average F1-score for the spam class.

In [26]:
spam_pipeline = Pipeline([
    ("text_features", TfidfVectorizer(stop_words="english")),
    ("spam_classifier", LogisticRegression(
        max_iter=1200,
        class_weight="balanced",
        random_state=7
    ))
])

In [27]:
search_space = {
    "text_features__max_features": [1500, 3000, 5000],
    "text_features__ngram_range": [(1, 1), (1, 2)],
    "spam_classifier__C": [0.25, 0.5, 1.0, 2.0]
}

In [28]:
spam_grid_search = GridSearchCV(
    estimator=spam_pipeline,
    param_grid=search_space,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

spam_grid_search.fit(X_train_normalized, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('text_features',
                                        TfidfVectorizer(stop_words='english')),
                                       ('spam_classifier',
                                        LogisticRegression(class_weight='balanced',
                                                           max_iter=1200,
                                                           random_state=7))]),
             n_jobs=-1,
             param_grid={'spam_classifier__C': [0.25, 0.5, 1.0, 2.0],
                         'text_features__max_features': [1500, 3000, 5000],
                         'text_features__ngram_range': [(1, 1), (1, 2)]},
             return_train_score=True, scoring='f1')

In [29]:
print("Best parameters:")
print(spam_grid_search.best_params_)

print("\nBest average validation F1:")
print(spam_grid_search.best_score_)

Best parameters:
{'spam_classifier__C': 2.0, 'text_features__max_features': 1500, 'text_features__ngram_range': (1, 2)}

Best average validation F1:
0.9111534056043566


GridSearchCV selects the best parameter combination according to the average F1-score across the 5 folds.
After selecting the best parameters, GridSearchCV refits the best estimator on the full training set.
This final model is then evaluated on the test set.

In [30]:
grid_results = pd.DataFrame(spam_grid_search.cv_results_)

grid_summary = grid_results[[
    "param_text_features__max_features",
    "param_text_features__ngram_range",
    "param_spam_classifier__C",
    "mean_test_score",
    "std_test_score",
    "mean_train_score"
]].sort_values(by="mean_test_score", ascending=False)

grid_summary

,param_text_features__max_features,param_text_features__ngram_range,param_spam_classifier__C,mean_test_score,std_test_score,mean_train_score
19,1500,"(1, 2)",2.0,0.911153,0.014261,0.952269
20,3000,"(1, 1)",2.0,0.906753,0.018739,0.960299
21,3000,"(1, 2)",2.0,0.903864,0.018509,0.955082
18,1500,"(1, 1)",2.0,0.902802,0.017600,0.956276
23,5000,"(1, 2)",2.0,0.896292,0.016089,0.958092
22,5000,"(1, 1)",2.0,0.895482,0.021084,0.963102
12,1500,"(1, 1)",1.0,0.895214,0.017081,0.933262
13,1500,"(1, 2)",1.0,0.893718,0.021669,0.930444
15,3000,"(1, 2)",1.0,0.888453,0.017182,0.937666
14,3000,"(1, 1)",1.0,0.886143,0.015403,0.935557


## Final Model Evaluation

After selecting the best configuration using GridSearchCV, we evaluate the final model on the test set.

In [31]:
final_spam_classifier = spam_grid_search.best_estimator_

final_predictions = final_spam_classifier.predict(X_test_normalized)

In [32]:
print("Final Confusion Matrix:")
print(confusion_matrix(y_test, final_predictions))

print("\nFinal Classification Report:")
print(classification_report(
    y_test,
    final_predictions,
    target_names=["ham", "spam"]
))

final_spam_f1 = f1_score(y_test, final_predictions, pos_label=1)
print("Final F1 score for spam:", final_spam_f1)

Final Confusion Matrix:
[[945  21]
 [  7 142]]

Final Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99       966
        spam       0.87      0.95      0.91       149

    accuracy                           0.97      1115
   macro avg       0.93      0.97      0.95      1115
weighted avg       0.98      0.97      0.98      1115

Final F1 score for spam: 0.9102564102564102


In [33]:
final_first_predictions = pd.DataFrame({
    "sms_message": X_test.iloc[:5].values,
    "true_value": y_test.iloc[:5].values,
    "predicted_value": final_predictions[:5]
})

final_first_predictions["true_label"] = final_first_predictions["true_value"].replace({
    0: "ham",
    1: "spam"
})

final_first_predictions["predicted_label"] = final_first_predictions["predicted_value"].replace({
    0: "ham",
    1: "spam"
})

final_first_predictions

,sms_message,true_value,predicted_value,true_label,predicted_label
0,Ok...,0,0,ham,ham
1,"Yeah hopefully, if tyler can't do it I could m...",0,0,ham,ham
2,FREE entry into our å£250 weekly competition j...,1,1,spam,spam
3,Lmao you know me so well...,0,0,ham,ham
4,I.ll hand her my phone to chat wit u,0,0,ham,ham


## Error Analysis

In this section, we inspect examples where the final model made wrong predictions.
This helps us understand the limitations of the classifier.

In [34]:
test_analysis = pd.DataFrame({
    "sms_text": X_test.values,
    "true_label": y_test.values,
    "predicted_label": final_predictions
})

test_analysis["true_name"] = test_analysis["true_label"].replace({
    0: "ham",
    1: "spam"
})

test_analysis["predicted_name"] = test_analysis["predicted_label"].replace({
    0: "ham",
    1: "spam"
})

wrong_predictions = test_analysis[
    test_analysis["true_label"] != test_analysis["predicted_label"]
]

wrong_predictions.head(10)

,sms_text,true_label,predicted_label,true_name,predicted_name
15,Hey...Great deal...Farm tour 9am to 5pm $95/pa...,0,1,ham,spam
41,Check Out Choose Your Babe Videos @ sms.shsex....,1,0,spam,ham
75,Can you please send me my aunty's number,0,1,ham,spam
94,"Funny fact Nobody teaches volcanoes 2 erupt, t...",0,1,ham,spam
99,"Do you ever notice that when you're driving, a...",1,0,spam,ham
105,"Hi, Mobile no. &lt;#&gt; has added you in th...",0,1,ham,spam
142,dating:i have had two of these. Only started a...,1,0,spam,ham
214,We made it! Eta at taunton is 12:30 as planned...,0,1,ham,spam
248,Missed call alert. These numbers called but le...,1,0,spam,ham
279,\Hello-/@drivby-:0quit edrunk sorry iff pthis ...,0,1,ham,spam


## Feature Importance

Logistic Regression learns a weight for each word or phrase.
Positive weights indicate features that increase the probability of spam.
Negative weights indicate features that are more related to ham messages.

In [35]:
final_vectorizer = final_spam_classifier.named_steps["text_features"]
final_logistic_model = final_spam_classifier.named_steps["spam_classifier"]

feature_names = final_vectorizer.get_feature_names_out()
feature_weights = final_logistic_model.coef_[0]

importance_table = pd.DataFrame({
    "word_or_phrase": feature_names,
    "model_weight": feature_weights
})

In [36]:
top_spam_indicators = importance_table.sort_values(
    by="model_weight",
    ascending=False
).head(20)

top_spam_indicators

,word_or_phrase,model_weight
816,number,7.987987
1361,url,6.602865
741,mobile,4.011917
1256,text,3.867666
1057,reply,3.834145
1339,uk,3.819441
1330,txt,3.697850
1089,sale,3.539932
1121,service,3.504243
1070,ringtone,3.347252


In [37]:
top_ham_indicators = importance_table.sort_values(
    by="model_weight",
    ascending=True
).head(20)

top_ham_indicators

,word_or_phrase,model_weight
920,ok,-2.124229
677,lt,-1.994853
457,gt,-1.894320
448,got,-1.748812
679,lt gt,-1.738075
506,hey,-1.502806
524,home,-1.498767
435,going,-1.485026
1462,work,-1.445867
297,don,-1.387869


## Final Conclusion

In this assignment, we built a binary SMS spam classifier using Logistic Regression.

First, we loaded and explored the SMS dataset.
Then we normalized the text and converted the messages into numerical features.
We tested different feature engineering options, including CountVectorizer and TF-IDF.

The main evaluation metric was F1-score for the spam class, because spam is the important class and the dataset is imbalanced.

Using GridSearchCV with 5-fold cross validation, we compared different combinations of:
- number of text features
- unigram and bigram representations
- Logistic Regression C values

Finally, we selected the best model according to validation F1-score and evaluated it on the test set.
We also analyzed wrong predictions and inspected the most important words learned by the Logistic Regression model.